# Voice Cleaning Pipeline - Custom DSP Modules Demo

This notebook demonstrates the 5 custom DSP modules:
1. **Audio Quality Profiler** - Analyzes input audio
2. **Adaptive Router** - Selects optimal processing method
3. **Spectral Restoration** - Restores lost frequencies
4. **Audio Quality Metrics** - Evaluates quality
5. **Optimized Utilities** - Performance benchmarks

## Setup and Installation

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install librosa pywt noisereduce numba matplotlib seaborn soundfile

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path (adjust if needed for Kaggle)
sys.path.insert(0, './src')

# Import custom modules
from audio_quality_profiler import AudioQualityProfiler
from audio_quality_metrics import AudioQualityMetrics
from spectral_restoration import SpectralRestoration
from adaptive_router import AdaptiveRouter
from optimized_utils import VectorizedAudioProcessor

print("✓ All modules imported successfully")

## Generate Test Audio

We'll create synthetic audio with known characteristics for demonstration.

In [ ]:
def generate_test_audio(duration=5.0, sample_rate=16000, noise_level=0.3):
    """Generate synthetic speech-like audio with noise"""
    t = np.linspace(0, duration, int(sample_rate * duration))
    
    # Generate speech-like signal (multiple harmonics)
    fundamental = 150  # Hz (typical male voice)
    signal = np.zeros(len(t))
    for harmonic in range(1, 8):
        amplitude = 1.0 / harmonic
        signal += amplitude * np.sin(2 * np.pi * fundamental * harmonic * t)
    
    # Add formants (resonances)
    signal += 0.3 * np.sin(2 * np.pi * 800 * t)  # F1
    signal += 0.2 * np.sin(2 * np.pi * 1200 * t)  # F2
    
    # Add noise
    noise = noise_level * np.random.randn(len(t))
    noisy_audio = signal + noise
    
    # Normalize
    noisy_audio = noisy_audio / np.max(np.abs(noisy_audio)) * 0.8
    signal = signal / np.max(np.abs(signal)) * 0.8
    
    return noisy_audio.astype(np.float32), signal.astype(np.float32), sample_rate

# Generate audio
noisy_audio, clean_audio, sample_rate = generate_test_audio(duration=5.0, noise_level=0.3)

print(f"Generated audio:")
print(f"  Duration: {len(noisy_audio) / sample_rate:.2f}s")
print(f"  Sample rate: {sample_rate} Hz")
print(f"  Samples: {len(noisy_audio):,}")

## Visualize Input Audio

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Waveform - Clean
time = np.arange(len(clean_audio)) / sample_rate
axes[0, 0].plot(time, clean_audio, linewidth=0.5)
axes[0, 0].set_title('Clean Signal (Reference)')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].grid(True, alpha=0.3)

# Waveform - Noisy
axes[0, 1].plot(time, noisy_audio, linewidth=0.5, color='orange')
axes[0, 1].set_title('Noisy Signal (Input)')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Amplitude')
axes[0, 1].grid(True, alpha=0.3)

# Spectrogram - Clean
from scipy import signal as sp_signal
f, t, Sxx = sp_signal.spectrogram(clean_audio, fs=sample_rate, nperseg=512)
axes[1, 0].pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10), shading='gouraud', cmap='viridis')
axes[1, 0].set_title('Spectrogram - Clean')
axes[1, 0].set_ylabel('Frequency (Hz)')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylim([0, 4000])

# Spectrogram - Noisy
f, t, Sxx = sp_signal.spectrogram(noisy_audio, fs=sample_rate, nperseg=512)
axes[1, 1].pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10), shading='gouraud', cmap='viridis')
axes[1, 1].set_title('Spectrogram - Noisy')
axes[1, 1].set_ylabel('Frequency (Hz)')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylim([0, 4000])

plt.tight_layout()
plt.show()

## 1. Audio Quality Profiler

Analyze the input audio to determine its characteristics and recommend processing.

In [ ]:
profiler = AudioQualityProfiler(sample_rate=sample_rate)
profile = profiler.profile_audio(noisy_audio, sample_rate)

print(profiler.format_profile(profile))

In [ ]:
# Visualize profile metrics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of key metrics
metrics_to_plot = {
    'SNR (dB)': profile['snr_db'],
    'Noise Floor (dB)': profile['noise_floor_db'],
    'Spectral Flatness': profile['spectral_flatness'] * 100,
    'ZCR': profile['zero_crossing_rate'] * 1000
}

axes[0].bar(metrics_to_plot.keys(), metrics_to_plot.values())
axes[0].set_title('Audio Profile Metrics')
axes[0].set_ylabel('Value')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# Recommendation
recommendation = profile['recommended_processing']
colors = {'light': 'green', 'medium': 'orange', 'heavy': 'red'}
axes[1].bar(['Recommended\nProcessing'], [1], color=colors[recommendation])
axes[1].set_title(f'Recommendation: {recommendation.upper()}')
axes[1].set_ylim([0, 1.2])
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

## 2. Adaptive Router

Route the audio to the appropriate processing method based on the profile.

In [ ]:
router = AdaptiveRouter(sample_rate=sample_rate)
processed_audio, method = router.route_processing(noisy_audio, profile)

print(f"Selected method: {router.get_method_description(method)}")
print(f"Output length: {len(processed_audio)} samples")

In [ ]:
# Visualize processing result
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Waveform comparison
axes[0].plot(time[:5000], noisy_audio[:5000], label='Noisy', alpha=0.7, linewidth=0.8)
axes[0].plot(time[:5000], processed_audio[:5000], label='Processed', alpha=0.7, linewidth=0.8)
axes[0].set_title(f'Waveform Comparison (Method: {method})')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Spectrogram comparison
f, t, Sxx_proc = sp_signal.spectrogram(processed_audio, fs=sample_rate, nperseg=512)
axes[1].pcolormesh(t, f, 10 * np.log10(Sxx_proc + 1e-10), shading='gouraud', cmap='viridis')
axes[1].set_title('Spectrogram - Processed')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylim([0, 4000])

plt.tight_layout()
plt.show()

## 3. Spectral Restoration

Restore high-frequency content that may have been lost during processing.

In [ ]:
restorer = SpectralRestoration(sample_rate=sample_rate)
restored_audio = restorer.adaptive_restoration(
    noisy_audio, processed_audio, strength='auto'
)

print(f"Restoration complete")
print(f"Output length: {len(restored_audio)} samples")

In [ ]:
# Visualize restoration effect
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

# Waveforms
axes[0].plot(time[:5000], processed_audio[:5000], label='Before Restoration', alpha=0.7)
axes[0].plot(time[:5000], restored_audio[:5000], label='After Restoration', alpha=0.7)
axes[0].set_title('Waveform - Restoration Effect')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Spectrogram - Before
f, t, Sxx_before = sp_signal.spectrogram(processed_audio, fs=sample_rate, nperseg=512)
axes[1].pcolormesh(t, f, 10 * np.log10(Sxx_before + 1e-10), shading='gouraud', cmap='viridis')
axes[1].set_title('Spectrogram - Before Restoration')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_ylim([0, 4000])

# Spectrogram - After
f, t, Sxx_after = sp_signal.spectrogram(restored_audio, fs=sample_rate, nperseg=512)
axes[2].pcolormesh(t, f, 10 * np.log10(Sxx_after + 1e-10), shading='gouraud', cmap='viridis')
axes[2].set_title('Spectrogram - After Restoration')
axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylim([0, 4000])

plt.tight_layout()
plt.show()

## 4. Audio Quality Metrics

Evaluate the quality of processed audio using 9 scientific metrics.

In [ ]:
metrics_calc = AudioQualityMetrics(sample_rate=sample_rate)

# Evaluate processed audio
metrics_processed = metrics_calc.comprehensive_evaluation(
    clean_audio, processed_audio, sample_rate
)

# Evaluate restored audio
metrics_restored = metrics_calc.comprehensive_evaluation(
    clean_audio, restored_audio, sample_rate
)

print("Metrics - Processed Audio:")
print(metrics_calc.format_results(metrics_processed))
print("\n" + "="*70 + "\n")
print("Metrics - Restored Audio:")
print(metrics_calc.format_results(metrics_restored))

In [ ]:
# Visualize metrics comparison
import pandas as pd

# Create comparison dataframe
metrics_names = [
    'SNR (dB)', 'PSNR (dB)', 'Seg. SNR (dB)',
    'LSD', 'IS Dist', 'Correlation',
    'Cep. Dist', 'Env. Dist', 'Quality Score'
]

processed_values = [
    metrics_processed['snr_db'],
    metrics_processed['psnr_db'],
    metrics_processed['segmental_snr_db'],
    metrics_processed['log_spectral_distance'],
    metrics_processed['itakura_saito_distance'],
    metrics_processed['correlation_coefficient'],
    metrics_processed['cepstral_distance'],
    metrics_processed['envelope_distance'],
    metrics_processed['overall_quality_score']
]

restored_values = [
    metrics_restored['snr_db'],
    metrics_restored['psnr_db'],
    metrics_restored['segmental_snr_db'],
    metrics_restored['log_spectral_distance'],
    metrics_restored['itakura_saito_distance'],
    metrics_restored['correlation_coefficient'],
    metrics_restored['cepstral_distance'],
    metrics_restored['envelope_distance'],
    metrics_restored['overall_quality_score']
]

df = pd.DataFrame({
    'Metric': metrics_names,
    'Processed': processed_values,
    'Restored': restored_values
})

print(df.to_string(index=False))

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_names))
width = 0.35

ax.bar(x - width/2, processed_values, width, label='Processed', alpha=0.8)
ax.bar(x + width/2, restored_values, width, label='Restored', alpha=0.8)

ax.set_xlabel('Metric')
ax.set_ylabel('Value')
ax.set_title('Quality Metrics Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Performance Benchmarks

Demonstrate CPU optimizations using Numba JIT compilation.

In [ ]:
processor = VectorizedAudioProcessor(sample_rate=sample_rate)

print(f"Numba available: {processor.numba_available}")
print("\nRunning benchmarks (this may take a minute)...\n")

benchmark_results = processor.benchmark_optimizations(noisy_audio, n_iterations=50)

print(processor.format_benchmark_results(benchmark_results))

In [ ]:
# Visualize speedups
operations = list(benchmark_results.keys())
speedups = [benchmark_results[op]['speedup'] for op in operations]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(operations, speedups, color=['#2ecc71', '#3498db', '#e74c3c'])
ax.set_ylabel('Speedup (x)')
ax.set_title('Performance Improvements with Numba JIT')
ax.set_ylim([0, max(speedups) * 1.2])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, speedup in zip(bars, speedups):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{speedup:.1f}x',
            ha='center', va='bottom', fontweight='bold')

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## Summary and Export

Save results to JSON for further analysis.

In [ ]:
import json

# Compile all results
results = {
    'audio_profile': profile,
    'processing_method': method,
    'metrics_processed': metrics_processed,
    'metrics_restored': metrics_restored,
    'benchmark_results': benchmark_results
}

# Save to JSON
with open('demo_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to demo_results.json")
print("\n" + "="*70)
print("DEMO COMPLETE")
print("="*70)
print(f"\nSummary:")
print(f"  Input SNR:              {profile['snr_db']:.1f} dB")
print(f"  Recommended processing: {profile['recommended_processing']}")
print(f"  Method used:            {method}")
print(f"  Quality (processed):    {metrics_processed['overall_quality_score']:.1f}/100")
print(f"  Quality (restored):     {metrics_restored['overall_quality_score']:.1f}/100")
print(f"  Avg speedup:            {np.mean([r['speedup'] for r in benchmark_results.values()]):.1f}x")